# Fusion with a fine-tuned (unfrozen) text encoder

Same architecture and data as the frozen-text fusion runs (15 log-sum-exp layer tokens + BAAI/bge-base-en-v1.5 question/answer token states, LayerNorm + linear 768→256, 2-layer fusion transformer, linear head on [CLS], BCE, batch 128, 40 epochs, best-source-validation-AUROC checkpoint, held-out TriviaQA). The only change: the bge-base encoder is inside the model and trained end to end (lr 2e-5; everything else lr 3e-4), with 0% text dropout — the best frozen setting (0.869 ± 0.003 TriviaQA AUROC over 3 seeds).

Runs: **fused, fine-tuned text encoder** with seeds 42 / 43 / 44, plus **text only, fine-tuned encoder** (seed 42). The fused model is also scored with each modality masked at test time.

In [ ]:
!git clone https://github.com/brainardphilemon/SALA.git /kaggle/working/SALA
%cd /kaggle/working/SALA
!git checkout 4c351e3
!pip install -q -r requirements-kaggle.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from pathlib import Path
SOURCE_NAME = 'sala_qwen25_g14_6src_triviaqa_v1'
domains = ['tqa', 'nq_open', 'sciq', 'popqa', 'web_questions', 'hotpotqa', 'triviaqa']
artifact_dirs = [p for p in Path('/kaggle/input').rglob(SOURCE_NAME) if p.name == SOURCE_NAME and (p / 'generation').is_dir()]
if len(artifact_dirs) != 1: raise RuntimeError(f'Expected one saved artifact set; found {artifact_dirs}')
ARTIFACT_DIR = artifact_dirs[0]
required = [*(ARTIFACT_DIR / 'generation' / d / f for d in domains for f in ('rows.jsonl', 'features.npy', 'manifest.json')), *(ARTIFACT_DIR / 'bleurt' / d / f for d in domains for f in ('scores.jsonl', 'manifest.json'))]
missing = [str(p) for p in required if not p.is_file()]
if missing: raise RuntimeError('Missing artifact files: ' + ', '.join(missing))
print('Saved features:', ARTIFACT_DIR)

In [ ]:
import subprocess, sys
CONFIGS = {'fused_ft': ['--mode', 'fused', '--text-dropout', '0', '--finetune-text-encoder'], 'text_ft': ['--mode', 'text', '--finetune-text-encoder']}
RUNS = [('fused_ft', 42), ('fused_ft', 43), ('fused_ft', 44), ('text_ft', 42)]
OUTPUT_DIRS = {}
for name, seed in RUNS:
    output_dir = Path(f'/kaggle/working/finetuned_text_{name}_s{seed}')
    OUTPUT_DIRS[(name, seed)] = output_dir
    command = [sys.executable, '/kaggle/working/SALA/sala_vlm_frozen_text.py', '--input-artifact-dir', str(ARTIFACT_DIR), '--output-dir', str(output_dir), '--bleurt-threshold', '0.5', '--epochs-erm', '40', '--seed', str(seed), '--train-device', 'cuda:0', '--batch-size', '128', '--image', 'logsumexp', '--text-encoder-lr', '2e-5', *CONFIGS[name]]
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, check=True)

In [ ]:
import json, pandas as pd
rows = []
for (name, seed), output_dir in OUTPUT_DIRS.items():
    s = json.loads((output_dir / 'summary.json').read_text())
    rows.append({'config': name, 'seed': seed, 'best_epoch': s['best_epoch'],
                 'source_val_auroc': s['best_source_validation_auroc_during_training'],
                 'source_test_auroc': s['source_test_metrics']['auroc'],
                 'target_auroc': s['target_metrics']['auroc'],
                 'target_balanced_accuracy': s['target_metrics']['balanced_accuracy'],
                 'text_masked_target_auroc': s.get('text_masked_target_auroc'),
                 'image_masked_target_auroc': s.get('image_masked_target_auroc')})
table = pd.DataFrame(rows)
display(table)
summary = table.groupby('config')[['source_val_auroc', 'source_test_auroc', 'target_auroc', 'target_balanced_accuracy', 'text_masked_target_auroc', 'image_masked_target_auroc']].agg(['mean', 'std'])
display(summary)
print(summary.to_string())